# Grid UCI BNN — Sampler Investigation

Loads **all available samplers** for one `(dataset, split)` and compares them across:
metrics, sparsity, calibration, posterior noise, ESS, and predictive intervals.

Adapted from `uci_investigate.ipynb` for the `gpu_friendly` tree: `grid_boomerang` / `grid_sticky_boomerang` / `nuts` / `nuts_horseshoe` instead of zigzag/boomerang x sticky x PLI, and noise is learned by every sampler here (no fixed-noise variant, so the old notebook's `LEARNED_NOISE` filter and `target_cache` keyed by bool are gone).

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
from torch import Tensor
from torch.distributions import Normal

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 1. Choose (dataset, split)

In [ ]:
RESULTS_DIR = Path("results/grid/uci_bnn_no_sigma_scale")

DATASET  = "boston"
SPLIT_ID = 0

# Display-name overrides (stem -> label)
LABELS = {
    "grid_boomerang":        "Grid Boomerang",
    "grid_sticky_boomerang": "Grid Sticky Boomerang",
    "nuts":                  "NUTS",
    "nuts_horseshoe":        "NUTS-HS",
}

# Colour palette -- one colour per base sampler family
COLORS = {
    "grid_boomerang":        "#FA5C00",
    "grid_sticky_boomerang": "#FA5C00",
    "nuts":                  "#0CCA38",
    "nuts_horseshoe":        "#0CCA38",
}

# Linestyle: solid for sticky/HS, dashed for vanilla
LINESTYLES = {
    "grid_boomerang":        "--",
    "grid_sticky_boomerang": "-",
    "nuts":                  "--",
    "nuts_horseshoe":        "-",
}

## 2. Load runs and reconstruct data split

In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, build_target, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)

split_dir = RESULTS_DIR / DATASET / f"split_{SPLIT_ID:02d}"
pt_files  = sorted(split_dir.glob("*.pt"))
print(split_dir)
assert pt_files, f"No .pt files found in {split_dir}."

print(f"Found {len(pt_files)} run(s) in {split_dir}:")
for p in pt_files:
    print(f"  {p.stem}")

In [ ]:
print("Loading raw dataset for test-set reconstruction...")
raw = load_raw_datasets((DATASET,))
X_all, y_all = raw[DATASET]

data   = make_split(X_all, y_all, seed=BASE_SEED + SPLIT_ID, dtype=DTYPE, device=DEVICE)
X_test = data["X_test"]
y_test = data["y_test"]
y_std  = data["y_std"]
print(f"  X_test: {X_test.shape}   y_std: {y_std:.4f}")

## 3. Rebuild target and compute predictions

One `BayesianModule` is built (shared by every run in this split -- they all share the same architecture/config, mirroring `uci_bnn_grid.py`'s `run_split`). Predictions go through `bm.module`/`bm.param_dict_fn` via `functional_call`, replacing the old tree's `TorchTarget`/`ModuleGaussianLikelihood.predict()`.

In [ ]:
bm = None

@torch.no_grad()
def predict_all(bm, weight_samples: Tensor, X_new: Tensor) -> Tensor:
    return torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_new,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]


runs: dict[str, dict] = {}   # stem -> payload

for pt in pt_files:
    stem = pt.stem
    run  = torch.load(pt, map_location="cpu", weights_only=False)

    if bm is None:
        cfg = BNNConfig(
            layer_sizes=run["layer_sizes"],
            activation=run["activation"],
            prior_sigma_scale=run["prior_sigma_scale"],
        )
        print(f"  Building target (layer_sizes={run['layer_sizes']}, act={run['activation']})...")
        bm, _, _ = build_target(data, cfg)
        print(f"    D = {bm.D}")

    samples = run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
    weight_samples = samples[:, :-1]

    preds     = predict_all(bm, weight_samples, X_test)   # [S, N]
    mean_pred = preds.mean(0)
    epist_std = preds.std(0)

    noise_samples = samples[:, -1].exp()       # [S]
    noise_std_eff = float(noise_samples.mean())

    total_std = (epist_std ** 2 + noise_std_eff ** 2).sqrt()

    runs[stem] = dict(
        run            = run,
        samples        = samples,
        weight_samples = weight_samples,
        mean_pred      = mean_pred,
        epist_std      = epist_std,
        total_std      = total_std,
        noise_std_eff  = noise_std_eff,
        noise_samples  = noise_samples,
        label          = LABELS.get(stem, stem),
        color          = COLORS.get(stem, "grey"),
        ls             = LINESTYLES.get(stem, "-"),
    )
    print(f"  [{stem}] done — {samples.shape[0]} samples")

sampler_order = list(runs.keys())

In [ ]:
# MAP prediction from x_ref (Adam estimate) -- one evaluation per run to verify x_ref consistency
print("MAP predictions (x_ref):")
for stem, s in runs.items():
    x_ref = s["run"]["x_ref"]
    if x_ref is None:
        print(f"  {s['label']}: no x_ref stored")
        continue
    x_ref = x_ref.to(dtype=DTYPE)
    weights_ref = x_ref[:-1]
    with torch.no_grad():
        pred = torch.func.functional_call(bm.module, bm.param_dict_fn(weights_ref), (X_test,)).squeeze(-1)
    noise = float(x_ref[-1].exp())
    rmse  = float(((pred - y_test) ** 2).mean().sqrt()) * y_std
    print(f"  {s['label']}: RMSE={rmse:.3f}  noise_std={noise:.3f}")

In [ ]:
for stem, s in runs.items():
    x_ref = s["run"]["x_ref"]
    if x_ref is None:
        continue
    x_ref = x_ref.to(dtype=DTYPE)
    ws = s["samples"].to(dtype=DTYPE)
    dists = (ws - x_ref).norm(dim=1)
    print(f"{s['label']}: mean dist={dists.mean():.3f}  std={dists.std():.3f}  max={dists.max():.3f}")

In [ ]:
_, x_ref_bm, Sigma_inv = build_target(data, BNNConfig(
    layer_sizes=list(runs.values())[0]["run"]["layer_sizes"],
    activation=list(runs.values())[0]["run"]["activation"],
    prior_sigma_scale=list(runs.values())[0]["run"]["prior_sigma_scale"],
))
si = Sigma_inv  # 1-D diagonal vector [D]

print(f"Shape : {si.shape}")
print(f"Min   : {si.min().item():.4g}")
print(f"Max   : {si.max().item():.4g}")
print(f"Ratio : {(si.max() / si.min()).item():.4g}")
print(f"Mean  : {si.mean().item():.4g}")
print(f"Median: {si.median().item():.4g}")

# Distribution of implied std devs (orbit radii per coordinate)
sigma_diag = (1.0 / si).sqrt()
print(f"\nImplied per-coord std (Sigma_sqrt diagonal):")
print(f"  Min   : {sigma_diag.min().item():.4g}")
print(f"  Max   : {sigma_diag.max().item():.4g}")
print(f"  Mean  : {sigma_diag.mean().item():.4g}")
print(f"  Median: {sigma_diag.median().item():.4g}")

In [ ]:
# Evaluate energy at x_ref, then take a few more Adam steps to sanity-check
# the MAP is actually near-converged.
e_ref = float(bm.energy(x_ref_bm))

beta = x_ref_bm.clone().requires_grad_(True)
opt = torch.optim.Adam([beta], lr=1e-3)
energies = []
for i in range(2000):
    opt.zero_grad()
    loss = bm.energy(beta)
    loss.backward()
    opt.step()
    if i % 200 == 0:
        energies.append(loss.item())

print(f"Energy at x_ref      : {e_ref:.4f}")
print(f"Energy after 2k more : {energies[-1]:.4f}")
print(f"Drop                 : {e_ref - energies[-1]:.4f}")
print(energies)

## 4. Metrics table

In [ ]:
from sazz.utils.metrics import ess_per_coord
import pandas as pd

def compute_rmse(y_true, mean_pred, y_std):
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std

def compute_nll(y_true, mean_pred, total_std, y_std):
    ll = (-0.5 * ((y_true - mean_pred) / total_std) ** 2
          - total_std.log() - 0.5 * math.log(2 * math.pi)).mean()
    return float(-ll + math.log(y_std))

def compute_crps(y_true, mean_pred, total_std, y_std):
    d = Normal(0.0, 1.0)
    sigma = total_std * y_std
    z = (y_true * y_std - mean_pred * y_std) / sigma
    return float((sigma * (z * (2*d.cdf(z) - 1) + 2*d.log_prob(z).exp()
                           - 1/math.sqrt(math.pi))).mean())

def compute_coverage(y_true, mean_pred, total_std, level=0.9):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_true - mean_pred).abs() <= z * total_std).float().mean())

rows = []
for stem, s in runs.items():
    ess         = ess_per_coord(s["weight_samples"])
    elapsed     = s["run"]["elapsed_sec"]
    grad_evals  = s["run"].get("gradient_evals")  # NUTS (num_steps) or PDMP; None if not tracked
    rows.append({
        "Sampler":    s["label"],
        "RMSE":       compute_rmse(y_test, s["mean_pred"], y_std),
        "NLL":        compute_nll(y_test, s["mean_pred"], s["total_std"], y_std),
        "CRPS":       compute_crps(y_test, s["mean_pred"], s["total_std"], y_std),
        "Cov 90%":    compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.90),
        "Cov 95%":    compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.95),
        "ESS min":    float(ess.min()),
        "ESS/s":      float(ess.min()) / elapsed,
        # ESS/gradient-eval: the fairest cross-family currency here -- PDMP
        # "iterations" and NUTS "draws" aren't comparable units, but both
        # samplers pay in gradient evaluations of the (log-)target.
        "ESS/grad":   (float(ess.min()) / grad_evals) if grad_evals else float("nan"),
        "Grad evals": grad_evals if grad_evals is not None else float("nan"),
        "Time (s)":   elapsed,
    })

metrics_df = pd.DataFrame(rows).set_index("Sampler")

def highlight_coverage(col):
    target = 0.90 if "90" in col.name else 0.95
    best = (col - target).abs().idxmin()
    return ["background-color: #68dc0f" if idx == best else "" for idx in col.index]

In [ ]:
metrics_df.style \
    .highlight_min(subset=["RMSE", "NLL", "CRPS"], color="#68dc0f") \
    .highlight_max(subset=["ESS/s", "ESS/grad"], color="#68dc0f") \
    .apply(highlight_coverage, subset=["Cov 90%", "Cov 95%"]) \
    .format(precision=5, na_rep="n/a")

## 4b. Sampler cost diagnostics (all samplers)

Compute-cost view across all four samplers, in a common currency (gradient evaluations of the log-target), since PDMP "skeleton events" and NUTS "draws" aren't directly comparable units -- both pay per gradient evaluation, so `Grad evals/s` and `ESS/grad` (section 4) are the fairest cross-family comparisons here. `t_max` columns are grid-sampler-only (NUTS has no adaptive horizon). Fields are only present in `.pt` files saved after `save_run`/NUTS runners started persisting them -- older runs show `n/a`; re-run the relevant sampler(s) to backfill.

In [ ]:
diag_rows = []
for stem, s in runs.items():
    r = s["run"]

    n_events   = r.get("n_events")
    elapsed    = r.get("elapsed_sec")
    grad_evals = r.get("gradient_evals")
    t_max_log  = r.get("grid_t_max_log")
    bound_viol = r.get("bound_violations")

    diag_rows.append({
        "Sampler":          s["label"],
        "Events/s":         (n_events / elapsed) if (n_events and elapsed) else float("nan"),
        "Grad evals/s":     (grad_evals / elapsed) if (grad_evals and elapsed) else float("nan"),
        "Grad evals/event": (grad_evals / n_events) if (grad_evals and n_events) else float("nan"),
        "t_max mean":       float(np.mean(t_max_log)) if t_max_log else float("nan"),
        "t_max min":        float(np.min(t_max_log)) if t_max_log else float("nan"),
        "t_max max":        float(np.max(t_max_log)) if t_max_log else float("nan"),
        "Bound violations": bound_viol if bound_viol is not None else float("nan"),
        "Time (s)":         elapsed if elapsed is not None else float("nan"),
    })

grid_diag_df = pd.DataFrame(diag_rows).set_index("Sampler")
grid_diag_df.style.format(precision=4, na_rep="n/a")

## 5. Sparsity profiles

Meaningful for `grid_sticky_boomerang`; the other samplers won't show real sparsity but are plotted for comparison.

In [ ]:
FREEZE_THR = 1e-3
window = 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for stem, s in runs.items():
    ws = s["weight_samples"]  # [S, D]

    # --- Sorted sparsity scatter ---
    sparsity = (ws.abs() < FREEZE_THR).float().mean(dim=0).numpy()
    sorted_sparsity = np.sort(sparsity)
    axes[0].scatter(range(len(sorted_sparsity)), sorted_sparsity,
                    alpha=0.5, color=s["color"], label=s["label"], s=8,
                    marker=("o" if s["ls"] == "-" else "x"))

    # --- Model size over samples ---
    model_size = (ws.abs() >= FREEZE_THR).sum(dim=1).float()
    rolling_mean = model_size.unfold(0, window, 1).mean(dim=1)
    axes[1].plot(model_size.numpy(), alpha=0.1, color=s["color"])
    axes[1].plot(range(window // 2, len(rolling_mean) + window // 2),
                 rolling_mean.numpy(), color=s["color"], ls=s["ls"],
                 lw=1.6, label=s["label"])

    print(f"{s['label']}: sparsity={sparsity.mean():.3f}, "
          f"mean active params={model_size.mean():.1f} / {ws.shape[1]}")

axes[0].set_xlabel("Parameter rank (sorted by sparsity)")
axes[0].set_ylabel("P(|param| < threshold)")
axes[0].set_title(f"{DATASET.capitalize()} — per-parameter sparsity (sorted)")
axes[0].legend(fontsize=8)

axes[1].set_xlabel("Sample")
axes[1].set_ylabel("# active params")
axes[1].set_title(f"{DATASET.capitalize()} — model size over samples")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Posterior noise

Every run learns noise, so this section always has data (no `if ln_runs:` guard needed, unlike the old notebook).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for stem, s in runs.items():
    ns = s["noise_samples"].numpy()
    ax.hist(ns, bins=60, density=True, alpha=0.45,
            color=s["color"], histtype="stepfilled", edgecolor="none")
    ax.axvline(ns.mean(), color=s["color"], lw=1.8, ls=s["ls"],
               label=f"{s['label']} (mean={ns.mean():.3f})")
ax.set_xlabel(r"$\sigma$ (standardised scale)")
ax.set_ylabel("Density")
ax.set_title(f"{DATASET.capitalize()} — posterior noise")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()